# Homework 1 — Fuzzy pose regulation & Wang–Mendel rule extraction

| | Exercise | Points |
|---|---|---|
| **1** | Fuzzy pose regulation of a mobile robot: debug, understand and tune a fuzzy controller | 55 |
| **2** | Learning fuzzy rules from data with the Wang–Mendel method (Mackey–Glass time series) | 45 |

**What to hand in:** this notebook, executed, with every `TODO` completed and every
question marked **Q** answered in the markdown cell beneath it. The manual part of
Exercise 2 may be written on paper. In that case, attach a photo or scan.

**Rules of the game**
- Run the notebook from top to bottom before submitting (*Kernel → Restart & Run all*).
  It must run without errors.
- The cells marked *provided* hold helper code. Do not change them; read them!
- You may add as many cells as you want.
- Explain *why*, not only *what*. A correct plot with no interpretation earns half
  the points.

## 0. Setup *(provided)*

Works on **Google Colab** and on a **local** Jupyter installation.

In [ ]:
import sys, math, pathlib, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install "torch-fis[envs]"
else:
    try:
        importlib.import_module('fuzzy_torch')
    except ModuleNotFoundError:
        # Inside a checkout of the repository (examples/homework/): put the repo's
        # parent directory on sys.path. Otherwise: pip install "torch-fis[envs]"
        sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[2]))

import numpy as np
import torch
import matplotlib.pyplot as plt

import fuzzy_torch as ft
from fuzzy_torch import (FuzzyVariable, FuzzyRule, TriangularMF, MamdaniSystem,
                         TSKSystem, TSKConsequent, Product)
from fuzzy_torch.envs import ControlMode, rk4_step
from fuzzy_torch.envs.plants import DiffDriveRobot

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
PI = math.pi
print('fuzzy_torch', ft.__version__, '| torch', torch.__version__)

---
# Exercise 1 — Fuzzy pose regulation of a mobile robot (55 pts)

## 1.0 The problem

A differential-drive robot sits somewhere on the floor with some heading. We want it
to reach a **goal pose** $(x_g, y_g, \theta_g)$: the right **position** *and* the
right **heading**, as when docking at a charging station. This is called
**pose regulation** (or *parking*).

A former member of the lab left us a **fuzzy controller** for this task. *"It works,
I think, mostly"*, they said. Your job is to debug it, understand it and tune it.

## 1.1 Background *(read it, nothing to derive)*

**The robot.** Its pose is $(x, y, \theta)$. It is commanded with a forward speed $v$
and a yaw rate $\omega$, and, ideally, it moves as a **unicycle**:

$$ \dot x = v\cos\theta, \qquad \dot y = v\sin\theta, \qquad \dot\theta = \omega. $$

It cannot move sideways, so it cannot just slide to the goal: a parking manoeuvre needs
curved paths.

**Polar coordinates.** Controllers for this task do not use $x, y, \theta$ directly.
They describe the error with three numbers, with $\Delta x = x_g - x$ and
$\Delta y = y_g - y$:

$$
\rho = \sqrt{\Delta x^2 + \Delta y^2}, \qquad
\alpha = \operatorname{atan2}(\Delta y, \Delta x) - \theta, \qquad
\beta = \theta_g - \theta - \alpha,
$$

with $\alpha$ and $\beta$ **always wrapped to $(-\pi, \pi]$**.

- $\rho$ is the **distance** to the goal.
- $\alpha$: *"how much must I turn to face the goal?"* It is the angle from the robot's
  heading to the line of sight to the goal.
- $\beta$: *"once there, how much will I still be misaligned?"* It is the angle from the
  line of sight to the goal heading.

At the goal, $\rho = \alpha = \beta = 0$. **Exactly at the goal, $\alpha$ and $\beta$
are undefined**, because the line of sight has no direction there. A real controller
therefore stops once $\rho$ is below a small tolerance.

**How a controller uses them.** The usual structure, and the one of our fuzzy
controller, is

- **speed from distance**, $v = f(\rho)$: fast when far, slower when near, **zero at
  the goal**;
- **yaw rate from the angles**, $\omega = g(\alpha, \beta)$. It turns **towards** the
  goal: $\omega$ has the sign of $\alpha$, and $\alpha$ dominates. It also turns
  **against** $\beta$, so that the path curves and arrives with the right heading.
  When the robot already faces the goal ($\alpha \approx 0$), $\omega$ must have the
  **opposite** sign of $\beta$.

`to_polar` below (provided) computes $(\rho, \alpha, \beta)$ correctly.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))
robot, th_r = np.array([0.4, 0.5]), np.radians(15)
goalp, th_g = np.array([3.2, 2.3]), np.radians(80)
los = np.arctan2(*(goalp - robot)[::-1])
pts = np.array([[0.3, 0], [-0.18, 0.15], [-0.18, -0.15]])
for p, th, c, al in [(robot, th_r, 'C0', 1.0), (goalp, th_g, 'k', 0.4)]:
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    ax.add_patch(plt.Polygon(pts @ R.T + p, color=c, alpha=al))
ax.plot(*zip(robot, goalp), 'C3--', lw=1.2)
ax.annotate(r'$\rho$', (robot + goalp) / 2 + [-0.15, 0.12], color='C3', fontsize=13)
for p, th, c in [(robot, th_r, 'C0'), (goalp, th_g, 'k'), (goalp, los, 'C3')]:
    ax.plot([p[0], p[0] + 0.9 * np.cos(th)], [p[1], p[1] + 0.9 * np.sin(th)], color=c, lw=1)
a = np.linspace(th_r, los, 30)
ax.plot(robot[0] + 0.65 * np.cos(a), robot[1] + 0.65 * np.sin(a), 'C2')
ax.annotate(r'$\alpha$', robot + 0.75 * np.array([np.cos(a[15]), np.sin(a[15])]), color='C2', fontsize=13)
b = np.linspace(los, th_g, 30)
ax.plot(goalp[0] + 0.6 * np.cos(b), goalp[1] + 0.6 * np.sin(b), 'C1')
ax.annotate(r'$\beta$', goalp + 0.7 * np.array([np.cos(b[15]), np.sin(b[15])]), color='C1', fontsize=13)
ax.plot([robot[0], robot[0] + 1.2], [robot[1], robot[1]], 'k:', lw=0.8)
ax.annotate(r'$\theta$', robot + [0.95, 0.08], fontsize=11)
ax.plot([goalp[0], goalp[0] + 1.0], [goalp[1], goalp[1]], 'k:', lw=0.8)
ax.annotate(r'$\theta_g$', goalp + [0.45, 0.3], fontsize=11)
ax.text(robot[0] - 0.2, robot[1] - 0.35, 'robot', color='C0')
ax.text(goalp[0] - 0.3, goalp[1] - 0.4, 'goal', color='k')
ax.set_aspect('equal')
ax.set_xlim(-0.2, 4.6)
ax.set_ylim(-0.1, 3.4)
ax.set_title(r'Polar error coordinates ($\alpha>0$, $\beta>0$ here)')
plt.show()

### Simulation helpers *(provided)*

The plant is `fuzzy_torch.envs.plants.DiffDriveRobot`. It is a **dynamic** model with
mass, inertia, tyre friction and slip. It is not the ideal kinematic model. In
`ControlMode.VELOCITY` it has a fast inner loop that tracks a commanded forward speed
$v$ (m/s) and yaw rate $\omega$ (rad/s). The commands saturate at
$|v| \le 2$ m/s and $|\omega| \le 3$ rad/s, and they are tracked with a small lag
(≈ 0.1–0.2 s).

A **controller** is any callable `controller(state, goal) -> u` where

- `state` is a `(B, 6)` tensor: `x, y, psi, vx, vy, omega`. Only the first three,
  the pose, matter here. `psi` is the heading $\theta$ in rad, as integrated by
  odometry, so **it is not wrapped** to $(-\pi, \pi]$.
- `goal` is a `(3,)` tensor $(x_g, y_g, \theta_g)$.
- `u` is a `(B, 2)` tensor of commands $(v, \omega)$.

`B` is the batch size: all start poses are simulated at once.

In [ ]:
DT_CONTROL = 0.05            # the controller runs at 20 Hz; the plant at 100 Hz (RK4)
GOAL = (0.0, 0.0, 0.0)


def wrap(angle):
    """Wrap an angle (tensor) to (-pi, pi]."""
    return torch.atan2(torch.sin(angle), torch.cos(angle))


def simulate(controller, starts, goal=GOAL, T=25.0, robot=None):
    """Closed-loop simulation. starts: (B, 3) poses. Returns states (B, N+1, 6), commands (B, N, 2)."""
    robot = robot or DiffDriveRobot(mode=ControlMode.VELOCITY)
    starts = torch.as_tensor(starts, dtype=torch.float32).reshape(-1, 3)
    goal = torch.as_tensor(goal, dtype=torch.float32)
    state = torch.zeros(starts.shape[0], 6)
    state[:, :3] = starts
    lo, hi = robot.action_bounds()
    substeps = round(DT_CONTROL / robot.default_dt)
    states, commands = [state], []
    with torch.no_grad():
        for _ in range(round(T / DT_CONTROL)):
            u = torch.max(torch.min(controller(state, goal), hi), lo)   # actuator limits
            for _ in range(substeps):
                state = rk4_step(robot.derivatives, state, robot.map_action(u, state),
                                 robot.default_dt)
            states.append(state)
            commands.append(u)
    return torch.stack(states, 1), torch.stack(commands, 1)


def draw_robot(ax, x, y, th, color, size=0.22, alpha=1.0):
    pts = np.array([[size, 0], [-0.6 * size, 0.5 * size], [-0.6 * size, -0.5 * size]])
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    ax.add_patch(plt.Polygon(pts @ R.T + [x, y], color=color, alpha=alpha, zorder=3))


def plot_trajectories(S, goal=GOAL, ax=None, title=None, upto=None):
    """Top view of every trajectory in S (B, N+1, 6): o = start, triangles = robot."""
    ax = ax or plt.subplots(figsize=(5.5, 5.5))[1]
    S = S.detach().numpy()
    upto = S.shape[1] if upto is None else upto
    colors = plt.cm.tab10(np.arange(len(S)) % 10)
    for s, c in zip(S, colors):
        ax.plot(s[:upto, 0], s[:upto, 1], color=c, lw=1.4)
        ax.plot(s[0, 0], s[0, 1], 'o', color=c, ms=4)
        draw_robot(ax, s[0, 0], s[0, 1], s[0, 2], c, alpha=0.35)
        draw_robot(ax, *s[upto - 1, :3], c)
    draw_robot(ax, goal[0], goal[1], goal[2], 'k', size=0.3, alpha=0.25)
    ax.plot(goal[0], goal[1], 'k+', ms=12, mew=2)
    ax.set_aspect('equal')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    if title:
        ax.set_title(title)
    return ax


def polar_errors(S, goal=GOAL):
    """rho, alpha, beta along trajectories (computed here with *correct* wrapping)."""
    goal = torch.as_tensor(goal, dtype=torch.float32)
    dx, dy = goal[0] - S[..., 0], goal[1] - S[..., 1]
    rho = torch.hypot(dx, dy)
    alpha = wrap(torch.atan2(dy, dx) - S[..., 2])
    beta = wrap(goal[2] - S[..., 2] - alpha)
    return rho, alpha, beta


def plot_signals(S, U, goal=GOAL, which=0, title=None):
    """Errors and commands over time for trajectory number `which`."""
    rho, alpha, beta = (e[which].numpy() for e in polar_errors(S, goal))
    t = np.arange(S.shape[1]) * DT_CONTROL
    heading_err = wrap(torch.as_tensor(goal[2]) - S[which, :, 2]).numpy()
    fig, axs = plt.subplots(1, 3, figsize=(13, 3))
    axs[0].plot(t, rho)
    axs[0].set_title(r'distance $\rho$ [m]')
    axs[1].plot(t, np.degrees(alpha), label=r'$\alpha$')
    axs[1].plot(t, np.degrees(beta), label=r'$\beta$')
    axs[1].plot(t, np.degrees(heading_err), 'k:', label=r'$\theta_g-\theta$')
    axs[1].set_title('angles [deg]')
    axs[1].legend()
    axs[2].plot(t[1:], U[which, :, 0].numpy(), label='v [m/s]')
    axs[2].plot(t[1:], U[which, :, 1].numpy(), label=r'$\omega$ [rad/s]')
    axs[2].set_title('commands')
    axs[2].legend()
    for ax in axs:
        ax.set_xlabel('t [s]')
    if title:
        fig.suptitle(title)
    fig.tight_layout()


def pose_metrics(S, goal=GOAL, pos_tol=0.05, ang_tol_deg=5.0):
    """Per-trajectory metrics at the end of the simulation."""
    goal = torch.as_tensor(goal, dtype=torch.float32)
    rho, _, _ = polar_errors(S, goal)
    heading = wrap(goal[2] - S[..., 2]).abs() * 180 / PI
    ok = (rho < pos_tol) & (heading < ang_tol_deg)
    # time to goal: first instant after which the robot stays within tolerance
    stays = torch.flip(torch.cumprod(torch.flip(ok, [1]).int(), 1), [1]).bool()
    t_goal = [(stays[i].int().argmax().item() * DT_CONTROL) if stays[i, -1] else float('nan')
              for i in range(len(S))]
    path = (S[:, 1:, :2] - S[:, :-1, :2]).norm(dim=-1).sum(1)
    return {'reached': ok[:, -1].tolist(), 'final_rho': rho[:, -1].tolist(),
            'final_heading_err_deg': heading[:, -1].tolist(), 'time_to_goal': t_goal,
            'path_length': path.tolist()}


def print_metrics(name, m):
    n = len(m['reached'])
    print(f"{name}: reached {sum(m['reached'])}/{n} | "
          f"mean final rho {np.mean(m['final_rho']):.3f} m | "
          f"mean heading err {np.mean(m['final_heading_err_deg']):.1f} deg | "
          f"mean path {np.mean(m['path_length']):.2f} m | "
          f"mean time to goal {np.nanmean(m['time_to_goal']) if any(m['reached']) else float('nan'):.1f} s")


def animate(S, goal=GOAL, every=4, width=420):
    """Animated top view (needs Pillow). Display it as the last expression of a cell."""
    from fuzzy_torch.envs import Video
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    frames = []
    for k in list(range(1, S.shape[1], every)) + [S.shape[1]]:
        fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=80)
        FigureCanvasAgg(fig)
        plot_trajectories(S, goal, ax=ax, upto=k, title=f't = {(k - 1) * DT_CONTROL:4.1f} s')
        lim = float(S[..., :2].abs().max()) + 0.4
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        fig.canvas.draw()
        frames.append(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())
        plt.close(fig)
    return Video(frames, fps=1 / (every * DT_CONTROL) * 2, width=width)   # 2x speed


def circle_starts(n=8, radius=2.5):
    """n poses on a circle around the origin, tangent to it (counter-clockwise)."""
    ang = torch.arange(n) * 2 * PI / n
    return torch.stack([radius * torch.cos(ang), radius * torch.sin(ang), wrap(ang + PI / 2)], 1)


STARTS = circle_starts()
# Two extra start poses whose odometry heading has accumulated whole turns
# (the robot already rotated a lot in a previous mission).
ODOMETRY_STARTS = torch.tensor([[2.0, 1.0, 2 * PI + 0.5], [-2.0, 2.0, -2 * PI - 1.0]])

def to_polar(state, goal):
    """(rho, alpha, beta) of a batch of poses (B, >=3) w.r.t. goal (3,), angles wrapped."""
    goal = torch.as_tensor(goal, dtype=torch.float32)
    dx, dy = goal[0] - state[:, 0], goal[1] - state[:, 1]
    rho = torch.hypot(dx, dy)
    alpha = wrap(torch.atan2(dy, dx) - state[:, 2])
    beta = wrap(goal[2] - state[:, 2] - alpha)
    return rho, alpha, beta


_g = torch.Generator().manual_seed(0)
RANDOM_STARTS = torch.cat([6 * torch.rand(20, 2, generator=_g) - 3,
                           (2 * torch.rand(20, 1, generator=_g) - 1) * PI], 1)
ALL_STARTS = torch.cat([STARTS, RANDOM_STARTS])

### The controller we inherited *(provided — it is faulty)*

It is made of two Mamdani systems, following the structure of 1.1:

- a **speed** system $v = f(\rho)$, with 3 rules;
- a **yaw-rate** system $\omega = g(\alpha, \beta)$, with a $5\times5$ rule table.

We test it on eight start poses on a circle of radius 2.5 m around the goal
$(0, 0, 0)$, all of them tangent to the circle.

In [ ]:
def triangular_partition(var, names, centers):
    """Add triangular terms to `var` so that neighbouring terms cross at 0.5 (a 'strong' partition)."""
    for i, (name, c) in enumerate(zip(names, centers)):
        left = centers[i - 1] if i > 0 else c
        right = centers[i + 1] if i < len(centers) - 1 else c
        var.add_term(name, TriangularMF(left, c, right))
    return var


ANGLE_TERMS = ['NB', 'NS', 'Z', 'PS', 'PB']
ANGLE_CENTERS = [-PI, -PI / 4, 0.0, PI / 4, PI]


class FaultyPoseController:
    """Fuzzy pose controller left by a former lab member. It does not work properly."""

    def __init__(self):
        # ---- speed: v = f(rho)
        rho = triangular_partition(FuzzyVariable('rho', 0.0, 3.0),
                                   ['Zero', 'Near', 'Far'], [0.0, 1.2, 3.0])
        v = FuzzyVariable('v', -0.6, 2.0)
        v.add_term('Zero', TriangularMF(-0.6, 0.0, 0.6))
        v.add_term('Slow', TriangularMF(0.0, 0.6, 1.2))
        v.add_term('Fast', TriangularMF(0.6, 2.0, 2.0))
        self.speed = MamdaniSystem([rho], v, [
            FuzzyRule((rho, 'Zero'), (v, 'Slow')),
            FuzzyRule((rho, 'Near'), (v, 'Slow')),
            FuzzyRule((rho, 'Far'), (v, 'Fast')),
        ])

        # ---- yaw rate: omega = g(alpha, beta)
        alpha = triangular_partition(FuzzyVariable('alpha', -PI, PI), ANGLE_TERMS, ANGLE_CENTERS)
        beta = triangular_partition(FuzzyVariable('beta', -PI, PI), ANGLE_TERMS, ANGLE_CENTERS)
        omega = triangular_partition(FuzzyVariable('omega', -4.0, 4.0),
                                     ['NB', 'NM', 'NS', 'Z', 'PS', 'PM', 'PB'],
                                     [-3.0, -2.0, -0.8, 0.0, 0.8, 2.0, 3.0])
        #             beta:  NB    NS    Z     PS    PB
        table = {'NB':     ['NM', 'NB', 'NB', 'NB', 'NB'],
                 'NS':     ['PS', 'NS', 'NM', 'NB', 'NB'],
                 'Z':      ['NB', 'NS', 'Z',  'PS', 'PB'],    # alpha row
                 'PS':     ['PB', 'PB', 'PM', 'PS', 'NS'],
                 'PB':     ['PB', 'PB', 'PB', 'PB', 'PM']}
        self.table = table
        self.yaw = MamdaniSystem([alpha, beta], omega, [
            FuzzyRule([(alpha, a), (beta, b)], (omega, out))
            for a, row in table.items() for b, out in zip(ANGLE_TERMS, row)])

    def __call__(self, state, goal):
        dx, dy = goal[0] - state[:, 0], goal[1] - state[:, 1]
        rho = torch.hypot(dx, dy)
        alpha = torch.atan2(dy, dx) - state[:, 2]
        beta = goal[2] - state[:, 2] - alpha
        v = self.speed({'rho': rho.clamp(0, 3)})
        omega = self.yaw({'alpha': alpha.clamp(-PI, PI), 'beta': beta.clamp(-PI, PI)})
        return torch.stack([v, omega], -1)

In [ ]:
faulty = FaultyPoseController()
S_faulty, U_faulty = simulate(faulty, STARTS)
plot_trajectories(S_faulty, title='Faulty fuzzy controller')
plt.show()
print_metrics('faulty', pose_metrics(S_faulty))

In [ ]:
animate(S_faulty)

## 1.2 Debugging the fuzzy controller

**Q1.1 (25 pts).** `FaultyPoseController` contains **at least three faults**. For
**each** fault, give:
1. the **symptom**: what you see in the trajectories or signals; show a plot;
2. the **cause**: which line of `FaultyPoseController` is wrong, and why, with
   reference to the background in 1.1;
3. the **fix**.

Then complete `FuzzyPoseController`, a corrected copy of the faulty class. Keep it
fuzzy: fix the faults, but do not change the overall design.

*Hints:*
- `plot_signals(S, U, which=i)` shows $\rho, \alpha, \beta$ and the commands for one start
  pose. It computes the angles **correctly** (wrapped), so compare it with what
  the controller computes internally.
- Simulate `ODOMETRY_STARTS` as well, and compare path lengths and times, not only
  success.
- `controller.speed.plot_surface()` and `controller.yaw.plot_surface()` draw the control
  surfaces. `controller.yaw.plot_inference({'alpha': 0.0, 'beta': 0.5})` shows which
  rules fire for one situation.
- Fix one thing at a time and re-simulate after each change. That is how you *prove*
  what each fault does.
- To stop at the goal, `torch.where((rho < tol)[:, None], torch.zeros_like(u), u)`
  works on a whole batch.

In [ ]:
# TODO: experiments that isolate each fault (plots + metrics)

*Your diagnosis (symptom / cause / fix for each fault):*

In [ ]:
class FuzzyPoseController:
    """TODO: corrected copy of FaultyPoseController.

    Keep the arguments: `near` is the centre of the 'Near' set of rho, and `implication`
    is passed to both MamdaniSystems (None = Min). You will vary them in Q1.3.
    """

    def __init__(self, tol=0.01, near=1.2, implication=None):
        raise NotImplementedError

    def __call__(self, state, goal):
        raise NotImplementedError

In [ ]:
# TODO: show that each fix removes its symptom, then simulate FuzzyPoseController
#       on STARTS and ODOMETRY_STARTS (trajectories, signals, metrics, animation)

## 1.3 Understanding the fixed controller

**Q1.2 (12 pts).** Open up your `FuzzyPoseController`:
1. Plot the membership functions of all five variables ($\rho, v, \alpha, \beta, \omega$).
2. Plot both control surfaces. Where is the yaw-rate surface flat, and where is it
   steep? Where does it saturate?
3. Show the inference (which rules fire, with what strength, and the aggregated output
   set) for these two situations, and explain the resulting $\omega$ in words:
   - (a) the robot faces the goal, but the goal heading is rotated: $\alpha = 0$,
     $\beta = 1.0$;
   - (b) the goal is to the left, behind the robot: $\alpha = 2.5$, $\beta = -1.0$.
4. How many rules does the controller have in total? Why does the yaw-rate system need
   so many more than the speed system?

*Hints:* `variable.plot()`, `system.plot_surface()` and `system.plot_inference({...})`.
The variables live in `system.input_variables[name]` and `system.output_variable`.

In [ ]:
# TODO

*Your answers:*

## 1.4 Tuning and robustness

**Q1.3 (18 pts).** All experiments below use `ALL_STARTS` (the 8 circle starts plus
20 random starts). Report `pose_metrics` for each setting.
1. **Speed rules.** Simulate with the `Near` centre of $\rho$ at `near = 0.6`, `1.2`
   and `2.0`. How do success rate, final heading error and time to goal change? Explain
   the trade-off: why does a robot that approaches *too fast* arrive with the wrong
   heading?
2. **Implication.** Compare `implication=None` (Min, clipping) with
   `implication=Product()` (scaling), using the best `near`. Plot the yaw surfaces of
   both side by side. Is there a difference, and why is it small or large?
3. **Robustness.** The robot now carries a heavy payload and has worn tyres:
   `robot=DiffDriveRobot(mode=ControlMode.VELOCITY, mass=30.0, tire_friction=0.5)`.
   How does your best controller do? Show the trajectories. If it degrades, change the
   **rules or membership functions** (not the plant) to improve it, and explain what
   you changed.

*Hint:* `simulate(controller, ALL_STARTS, robot=...)` runs all 28 start poses in one
batch; it takes a few seconds.

In [ ]:
# TODO

*Your answers:*

**Q1.4 (bonus, up to 5 extra points).** With forward motion only, a robot whose goal is
*behind* it ($|\alpha| > \pi/2$) makes a long detour (see the first odometry start).
Extend `FuzzyPoseController` so that it may **drive backwards**.

*Hint:* if $|\alpha| > \pi/2$, compute the polar coordinates with the heading
$\theta + \pi$, use the same rules, and negate $v$. Watch out for chattering when
$|\alpha| \approx \pi/2$.

---
# Exercise 2 — Rule extraction with the Wang–Mendel method (45 pts)

In Exercise 1, an expert (a former lab member…) wrote the rules by hand. Often there is
no expert, but there is **data**. Can we **learn the rules from data**?

## 2.0 The data: Mackey–Glass time series *(provided)*

We use the chaotic **Mackey–Glass** series, a classic benchmark for fuzzy and neural
predictors (it is also used in the original Wang–Mendel paper):

$$ \dot x(t) = \frac{0.2\,x(t-\tau)}{1 + x(t-\tau)^{10}} - 0.1\,x(t), \qquad \tau = 17. $$

**Task:** predict the **next value** $x(t+1)$ from a window of the last six values,
$x(t-5), \dots, x(t)$. Each data entry is one input–output pair with six inputs and
one output:

$$ \big(\,x_1, x_2, x_3, x_4, x_5, x_6 \;;\; y\,\big)
   = \big(\,x(t-5),\, x(t-4),\, x(t-3),\, x(t-2),\, x(t-1),\, x(t) \;;\; x(t+1)\,\big). $$

The series is rounded to 3 decimals, so that your hand calculations and your code
give exactly the same numbers.

In [ ]:
def mackey_glass(n=1100, tau=17, dt=0.1, x0=1.2, discard=314):
    """Euler integration of the Mackey-Glass equation, sampled every 1 time unit."""
    lag, per = round(tau / dt), round(1 / dt)
    total = (n + discard) * per
    x = np.empty(total + lag + 1)
    x[:lag + 1] = x0
    for k in range(lag, total + lag):
        x[k + 1] = x[k] + dt * (0.2 * x[k - lag] / (1 + x[k - lag] ** 10) - 0.1 * x[k])
    return np.round(x[lag + discard * per::per][:n], 3)


def make_dataset(series, window=6, horizon=1):
    """Rows (x(t-window+1), ..., x(t)) -> x(t+horizon)."""
    t = np.arange(window - 1, len(series) - horizon)
    X = np.stack([series[t - lag] for lag in range(window - 1, -1, -1)], axis=1)
    return X, series[t + horizon]


series = mackey_glass()
X, y = make_dataset(series)
N_TRAIN = 500
X_train, y_train = X[:N_TRAIN], y[:N_TRAIN]
X_test, y_test = X[N_TRAIN:N_TRAIN + 500], y[N_TRAIN:N_TRAIN + 500]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(series, lw=1)
ax.axvspan(0, N_TRAIN + 6, color='C0', alpha=0.08, label='train')
ax.axvspan(N_TRAIN + 6, N_TRAIN + 506, color='C1', alpha=0.08, label='test')
ax.set_xlabel('t')
ax.set_ylabel('x(t)')
ax.set_title('Mackey-Glass series')
ax.legend()
plt.show()

print('The first 5 data entries:')
print(' k |    x1     x2     x3     x4     x5     x6  |   y')
for k in range(5):
    print(f'{k + 1:2d} | ' + '  '.join(f'{v:.3f}' for v in X[k]) + f' | {y[k]:.3f}')

## 2.1 The Wang–Mendel method

The **Wang–Mendel (WM) method** (L.-X. Wang & J. M. Mendel, *Generating fuzzy rules by
learning from examples*, IEEE Trans. SMC, 1992) turns a table of input–output examples
into a **fuzzy rule base** in a single pass over the data. It needs no iterations and no
gradients, and the result is a readable list of IF–THEN rules.

The idea: every example is a small piece of evidence of the form *"when the inputs
looked like this, the output looked like that"*. WM translates each example into
linguistic terms, which gives one rule per example. It then keeps the most convincing
rule wherever examples disagree.

Given $N$ examples $(x_1^{(k)}, \dots, x_m^{(k)}; y^{(k)})$, $k = 1..N$:

**Step 1 — Divide the domains into fuzzy regions.** For each variable, choose an
interval $[\,x^-, x^+]$ that covers its values and divide it into fuzzy sets. Here
every variable ($x_1, \dots, x_6$ and $y$) uses the **same** five triangular sets on
$[0.2, 1.4]$, named as in the paper (*S* = small, *CE* = centre, *B* = big):

| set | S2 | S1 | CE | B1 | B2 |
|---|---|---|---|---|---|
| centre $c$ | 0.2 | 0.5 | 0.8 | 1.1 | 1.4 |

Each triangle peaks at its centre and reaches zero at the neighbouring centres:

$$ \mu_c(v) = \max\!\left(0,\; 1 - \frac{|v - c|}{0.3}\right). $$

**Step 2 — Generate one rule from each example.** For each variable of the example,
compute its degree in every set, and assign the variable to the set with the
**maximum** degree. The example
$(x_1, \dots, x_6; y)$ then becomes the rule

$$ \text{IF } x_1 \text{ is } A_1 \text{ AND } \dots \text{ AND } x_6 \text{ is } A_6
   \text{ THEN } y \text{ is } C. $$

**Step 3 — Give each rule a degree.** Examples can generate conflicting rules, and some
rules are more convincing than others: an example that sits right at the centres of its
sets is a *typical* example, while one near the border between two sets is ambiguous.
WM measures this with the **degree of the rule**, the product of the memberships that
defined it:

$$ D(\text{rule}) = \mu_{A_1}(x_1)\,\mu_{A_2}(x_2)\cdots\mu_{A_6}(x_6)\;\mu_C(y). $$

(The paper also allows multiplying by an expert's confidence in the example; here it
is 1.)

**Step 4 — Build the rule base and resolve conflicts.** Two rules **conflict** if they
have the **same IF part** but a **different THEN part**. From each group of rules with
the same IF part, WM keeps only the rule with the **highest degree**. Identical rules
collapse into one. The surviving rules form the rule base. There is at most one rule
per combination of input sets, so at most $5^6$ here, and usually far fewer, because
the data does not visit every combination.

**Step 5 — Use the rule base.** For a new input $\mathbf x$, rule $i$ fires with
strength $w_i(\mathbf x) = \prod_j \mu_{A_{ij}}(x_j)$ (product AND). The output is the
**centre-average** of the rules' output centres $\bar y_i$:

$$ \hat y(\mathbf x) = \frac{\sum_i w_i(\mathbf x)\,\bar y_i}{\sum_i w_i(\mathbf x)}. $$

If no rule fires ($\sum_i w_i = 0$), the rule base has nothing to say about $\mathbf x$.

The five sets of Step 1 *(provided)*:

In [ ]:
WM_LABELS = ['S2', 'S1', 'CE', 'B1', 'B2']
WM_CENTERS = [0.2, 0.5, 0.8, 1.1, 1.4]
_demo = triangular_partition(FuzzyVariable('x', 0.2, 1.4), WM_LABELS, WM_CENTERS)
_demo.plot(value=X[0, 0])
plt.title(f'The five sets; marked: x1 of entry 1 = {X[0, 0]:.3f}')
plt.show()

### Q2.1 (15 pts) — Rule extraction by hand for the first 5 entries

Using **only pen, paper and a calculator** (not code), apply Steps 2–4 to the 5 data
entries printed in 2.0:

1. For every entry and every variable, give the winning set and its degree.
2. Write the 5 generated rules and their degrees $D$.
3. Identify the conflicts, resolve them, and write the final rule base.

*Your answer (or attach a photo of your paper). Write each cell as `set (degree)`:*

| k | $x_1$ | $x_2$ | $x_3$ | $x_4$ | $x_5$ | $x_6$ | $y$ | $D$ |
|---|---|---|---|---|---|---|---|---|
| 1 | | | | | | | | |
| 2 | | | | | | | | |
| 3 | | | | | | | | |
| 4 | | | | | | | | |
| 5 | | | | | | | | |

Conflicts:

Final rule base:

### Q2.2 (20 pts) — An automatic Wang–Mendel pipeline — TODO

Implement the method for **any number of inputs** and **any number of sets**, and use
`fuzzy_torch` for Step 5. Complete the four functions below:

| function | does |
|---|---|
| `make_partition(name, lo, hi, n_sets, labels=None)` | Step 1: a `FuzzyVariable` with `n_sets` evenly spaced triangles |
| `winning_sets(var, values)` | Step 2: the winning set name and its degree, for each value |
| `wang_mendel(X, y, in_vars, out_var)` | Steps 2–4: the rule base `{IF part: (THEN set, D)}` |
| `build_system(rules, in_vars, out_var)` | Step 5: a `fuzzy_torch` system that predicts |

Then:
1. **Check against your manual result.** Run your pipeline on the first 5 entries with
   the partition of 2.1. Its rules must equal `MY_MANUAL_RULES` (fill it in from Q2.1).
2. Extract the rules from `X_train` (500 entries), predict `X_test`, and report the
   **RMSE**. Plot the prediction against the truth, and print the 10 rules with the
   highest degree in readable form.
3. **Experiment** with the number of sets $n \in \{3, 5, 7, 9, 11, 15, 21, 31, 41\}$.
   Tabulate the number of rules, the test RMSE and the number of test entries where no
   rule fires. Plot the RMSE against $n$, together with the RMSE of the **naive
   forecast** $\hat x(t+1) = x(t)$ as a horizontal line.

*Hints (code):*
- `triangular_partition` from Exercise 1 already builds evenly spaced triangles if you
  give it `np.linspace(lo, hi, n_sets).tolist()` as centres.
- `var.fuzzify(torch.tensor(values))` returns `{set name: degrees}` for a whole array at
  once. Stack the degrees into an `(N, n_sets)` tensor; `.max(dim=-1)` gives the
  winning degree and its index. Wrap it in `torch.no_grad()` and call `.numpy()` at
  the end.
- Use a Python `dict` keyed by the IF part as a tuple of set names, e.g.
  `('S1', 'S1', 'S1', 'S1', 'S1', 'CE')`. Then keeping the strongest rule is a single
  `if`.
- Step 5 in `fuzzy_torch`: a `TSKSystem` whose rules have **constant** consequents,
  `TSKConsequent(0, coefficients=[centre])`, and product AND,
  `FuzzyRule(..., t_norm=Product())`, computes exactly the centre-average formula. The
  centre of an output set is `out_var.terms[name].b.item()` (the triangle's peak).
- For the train/test experiment, take the universe from the training data with a
  small margin, e.g. `X_train.min() - 0.05` to `X_train.max() + 0.05`.
- Where no rule fires, `fuzzy_torch` returns `NaN` on purpose. Use `np.isnan` to count
  these entries, and compute the RMSE on the others.
- A rule base with hundreds of rules is fine: inference over 500 test entries takes
  well under a second.

In [ ]:
def make_partition(name, lo, hi, n_sets, labels=None):
    """FuzzyVariable `name` on [lo, hi] with n_sets evenly spaced triangular sets."""
    # TODO
    raise NotImplementedError


def winning_sets(var, values):
    """For each value: (name of the set with the largest degree, that degree)."""
    # TODO
    raise NotImplementedError


def wang_mendel(X, y, in_vars, out_var):
    """Rule base {(set of x1, ..., set of xm): (set of y, degree)} after conflict resolution."""
    # TODO: Steps 2-4
    raise NotImplementedError


def build_system(rules, in_vars, out_var):
    """A fuzzy_torch system: product AND + centre-average output (Step 5)."""
    # TODO
    raise NotImplementedError


def predict(system, X):
    """Run `system` on an (N, n_inputs) array -> (N,) numpy array. (provided)"""
    names = system.input_order
    with torch.no_grad():
        return system({n: torch.as_tensor(X[:, j], dtype=torch.float32)
                       for j, n in enumerate(names)}).numpy()

**Step 1 — check against your manual result.** Fill in `MY_MANUAL_RULES` from Q2.1.

In [ ]:
MY_MANUAL_RULES = {
    # ('x1 set', 'x2 set', ..., 'x6 set'): ('y set', degree),
}

wm_vars = [triangular_partition(FuzzyVariable(f'x{j + 1}', 0.2, 1.4), WM_LABELS, WM_CENTERS)
           for j in range(6)]
wm_out = triangular_partition(FuzzyVariable('y', 0.2, 1.4), WM_LABELS, WM_CENTERS)
rules5 = wang_mendel(X[:5], y[:5], wm_vars, wm_out)
for ant, (cons, d) in rules5.items():
    print('IF', ' AND '.join(f'x{j + 1} is {t}' for j, t in enumerate(ant)),
          f'THEN y is {cons}   (D = {d:.3f})')

assert set(rules5) == set(MY_MANUAL_RULES), 'IF parts differ from your manual result'
for ant, (cons, d) in MY_MANUAL_RULES.items():
    assert rules5[ant][0] == cons and abs(rules5[ant][1] - d) < 2e-3, f'rule {ant} differs'
print('pipeline agrees with the manual extraction')

**Step 2 — train / test.**

In [ ]:
# TODO: partition from the training data, wang_mendel, build_system, RMSE, plots

**Step 3 — experiments.**

In [ ]:
# TODO: number of sets; table + plot

### Q2.3 (10 pts) — Discussion

1. How do the number of rules, the RMSE and the number of no-rule entries change with
   $n$? Explain why. From which $n$ on does WM beat the naive forecast, and why is the
   naive forecast so hard to beat here?
2. What is the largest possible number of rules with $n$ sets and a window of 6 inputs?
   How many did you get? What does this say about using WM with long windows?
3. Propose two ways to give a prediction for test entries where no rule fires.
4. WM keeps the rule with the highest degree when rules conflict. Give an example
   where this choice is bad (think of noise or outliers), and suggest an alternative.

**Bonus (up to 5 extra points).** The WM system is a `torch.nn.Module`. Use it as the
**initialisation** and fine-tune it with gradient descent (`torch.optim.Adam` on the
MSE over `X_train`, as in notebook 01). Report the RMSE before and after.

*Your discussion:*

In [ ]:
# Bonus

---
### References
- R. Siegwart, I. Nourbakhsh, D. Scaramuzza, *Introduction to Autonomous Mobile Robots*,
  MIT Press, 2nd ed., Section 3.6 (pose regulation in polar coordinates).
- L.-X. Wang, J. M. Mendel, *Generating fuzzy rules by learning from examples*,
  IEEE Trans. Systems, Man, and Cybernetics, 22(6), 1992.
- M. C. Mackey, L. Glass, *Oscillation and chaos in physiological control systems*,
  Science, 1977.